In [29]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')  # Suppress warnings for cleaner output

# Add project root to path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.models.polymer_gcn import PolymerGCN
from src.data.polymer_dataset import PolymerTgDataset

# Set up plotting defaults
plt.style.use('default')  # Use default style instead of seaborn
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['axes.grid'] = True
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Try to set seaborn style if available
try:
    import seaborn as sns
    sns.set_style("whitegrid")
    sns.set_palette("husl")
except Exception as e:
    print(f"Note: Could not set seaborn style ({str(e)}). Using default matplotlib style.")

# Enable better notebook formatting of outputs
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))


In [30]:
def diagnose_current_model(model_path, data_dir="data/processed", csv_file=None):
    """Run comprehensive diagnostics on current GCN
    
    Args:
        model_path (str): Path to the model checkpoint
        data_dir (str): Path to the processed data directory
        
    Returns:
        dict: Dictionary containing diagnostic results
    """
    # Load data and model
    dataset = PolymerTgDataset(
        root=data_dir,
        csv_file=csv_file,
        smiles_column='processed_smiles',  # Column name in the CSV file
        target_column='Tg',                # Column name in the CSV file
        split_type='all'                   # Use all data for diagnostics
    )
    model = PolymerGCN.load_from_checkpoint(model_path)
    
    print("=== GCN DIAGNOSTIC REPORT ===")
    
    # 1. Dataset Statistics
    print("\n1. DATASET ANALYSIS:")
    node_counts = [data.num_nodes for data in dataset]
    edge_counts = [data.num_edges for data in dataset]
    
    print(f"   • Total molecules: {len(dataset)}")
    print(f"   • Avg nodes per molecule: {np.mean(node_counts):.1f}")
    print(f"   • Avg edges per molecule: {np.mean(edge_counts):.1f}")
    print(f"   • Avg node degree: {np.mean(edge_counts) / np.mean(node_counts):.2f}")
    
    # 2. Feature Analysis
    sample_data = dataset[0]
    print(f"\n2. FEATURE ANALYSIS:")
    print(f"   • Node feature dim: {sample_data.x.shape[1]}")
    if hasattr(sample_data, 'edge_attr') and sample_data.edge_attr is not None:
        print(f"   • Edge feature dim: {sample_data.edge_attr.shape[1]}")
    else:
        print("   • No edge features found")
    print(f"   • Node features mean: {sample_data.x.mean():.3f}")
    print(f"   • Node features std: {sample_data.x.std():.3f}")
    print(f"   • Target mean: {sample_data.y.mean():.1f}")
    print(f"   • Target std: {sample_data.y.std():.1f}")
    
    # 3. Model Architecture
    print(f"\n3. MODEL ARCHITECTURE:")
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   • Total parameters: {total_params:,}")
    print(f"   • Trainable parameters: {trainable_params:,}")
    print(f"   • Model layers:")
    for name, module in model.named_children():
        print(f"     - {name}: {module.__class__.__name__}")
    
    # 4. Quick Prediction Test
    model.eval()
    with torch.no_grad():
        sample_batch = dataset[:10]  # Small batch
        predictions = model(sample_batch)
        
    print(f"\n4. PREDICTION ANALYSIS:")
    print(f"   • Prediction range: {predictions.min():.1f} to {predictions.max():.1f}")
    print(f"   • Target range: {sample_batch.y.min():.1f} to {sample_batch.y.max():.1f}")
    print(f"   • Predictions std: {predictions.std():.3f}")
    
    return {
        'dataset': dataset,
        'model': model,
        'node_counts': node_counts,
        'edge_counts': edge_counts,
        'predictions': predictions,
        'targets': sample_batch.y
    }


In [31]:
def plot_diagnostic_graphs(diagnostic_results):
    """Plot various diagnostic graphs from the model analysis
    
    Args:
        diagnostic_results (dict): Results from diagnose_current_model()
    """
    # Create a figure with multiple subplots
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
    
    # 1. Node Count Distribution
    ax1.hist(diagnostic_results['node_counts'], bins=30, alpha=0.7, color='skyblue')
    ax1.set_title('Distribution of Nodes per Molecule')
    ax1.set_xlabel('Number of Nodes')
    ax1.set_ylabel('Count')
    
    # 2. Edge Count Distribution
    ax2.hist(diagnostic_results['edge_counts'], bins=30, alpha=0.7, color='lightgreen')
    ax2.set_title('Distribution of Edges per Molecule')
    ax2.set_xlabel('Number of Edges')
    ax2.set_ylabel('Count')
    
    # 3. Node Degree Distribution
    degrees = [e/n for e, n in zip(diagnostic_results['edge_counts'], diagnostic_results['node_counts'])]
    ax3.hist(degrees, bins=30, alpha=0.7, color='salmon')
    ax3.set_title('Distribution of Average Node Degrees')
    ax3.set_xlabel('Average Node Degree')
    ax3.set_ylabel('Count')
    
    # 4. Predictions vs Targets
    ax4.scatter(diagnostic_results['targets'].numpy(), 
               diagnostic_results['predictions'].numpy(), 
               alpha=0.6)
    ax4.plot([min(diagnostic_results['targets']), max(diagnostic_results['targets'])],
             [min(diagnostic_results['targets']), max(diagnostic_results['targets'])],
             'r--', label='Perfect Prediction')
    ax4.set_title('Predictions vs Actual Values')
    ax4.set_xlabel('True Values')
    ax4.set_ylabel('Predicted Values')
    ax4.legend()
    
    plt.tight_layout()
    plt.show()


In [32]:
def analyze_layer_activations(model, dataset, num_samples=10):
    """Analyze the activations of each layer in the GCN
    
    Args:
        model (PolymerGCN): The model to analyze
        dataset (PolymerDataset): The dataset to use
        num_samples (int): Number of samples to analyze
        
    Returns:
        dict: Dictionary containing activation statistics for each layer
    """
    model.eval()
    activations = {}
    hooks = []
    
    # Register forward hooks for each layer
    def get_activation(name):
        def hook(model, input, output):
            activations[name] = output.detach()
        return hook
    
    # Add hooks to all GCN layers
    for name, module in model.named_modules():
        if 'conv' in name.lower():  # Only look at GCN convolution layers
            hooks.append(module.register_forward_hook(get_activation(name)))
    
    # Get activations for a batch of samples
    with torch.no_grad():
        sample_batch = dataset[:num_samples]
        _ = model(sample_batch)
    
    # Remove hooks
    for hook in hooks:
        hook.remove()
    
    # Analyze activations
    activation_stats = {}
    for name, activation in activations.items():
        if isinstance(activation, torch.Tensor):
            stats = {
                'mean': float(activation.mean()),
                'std': float(activation.std()),
                'min': float(activation.min()),
                'max': float(activation.max()),
                'zero_fraction': float((activation == 0).float().mean())
            }
            activation_stats[name] = stats
    
    return activation_stats

def plot_activation_stats(activation_stats):
    """Plot activation statistics for each layer
    
    Args:
        activation_stats (dict): Statistics from analyze_layer_activations()
    """
    num_layers = len(activation_stats)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Plot means and standard deviations
    layers = list(activation_stats.keys())
    means = [stats['mean'] for stats in activation_stats.values()]
    stds = [stats['std'] for stats in activation_stats.values()]
    
    x = range(len(layers))
    ax1.errorbar(x, means, yerr=stds, fmt='o-', capsize=5)
    ax1.set_title('Layer Activation Statistics')
    ax1.set_xlabel('Layer')
    ax1.set_ylabel('Activation Value')
    ax1.set_xticks(x)
    ax1.set_xticklabels(layers, rotation=45)
    
    # Plot zero fractions
    zero_fracs = [stats['zero_fraction'] for stats in activation_stats.values()]
    ax2.bar(x, zero_fracs)
    ax2.set_title('Fraction of Zero Activations')
    ax2.set_xlabel('Layer')
    ax2.set_ylabel('Fraction')
    ax2.set_xticks(x)
    ax2.set_xticklabels(layers, rotation=45)
    
    plt.tight_layout()
    plt.show()


In [33]:
# Example usage - replace with your paths
model_path = "path/to/your/model.ckpt"
data_dir = "data/processed"
csv_file = "/Volumes/Tanay-s T7/Codes/PolyGNN/data/processed/filtered_tg_dataset.csv"  # CSV file with SMILES and Tg data

# Run simplified diagnostics (bypasses dataset class issues)
try:
    results = diagnose_model_simple(model_path, csv_file)
    
    if results and results['model'] is not None:
        print("\n=== SUCCESS: Model loaded successfully ===")
        print("You can now run additional analysis if needed.")
    else:
        print("\n=== INFO: Data analysis completed, but model not loaded ===")
        print("Update the model_path variable with a valid checkpoint file.")
    
except Exception as e:
    print(f"Error running diagnostics: {str(e)}")


Error running diagnostics: name 'diagnose_model_simple' is not defined


In [34]:
def diagnose_model_simple(model_path, csv_file):
    """Simplified model diagnostics that bypasses dataset class issues
    
    Args:
        model_path (str): Path to the model checkpoint
        csv_file (str): Path to CSV file with SMILES and Tg data
        
    Returns:
        dict: Dictionary containing diagnostic results
    """
    import pandas as pd
    
    # Read CSV directly
    df = pd.read_csv(csv_file)
    print(f"=== SIMPLIFIED GCN DIAGNOSTIC REPORT ===")
    
    # 1. Dataset Analysis
    print(f"\n1. DATASET ANALYSIS:")
    print(f"   • Total samples: {len(df)}")
    print(f"   • Available columns: {list(df.columns)}")
    
    # Check for required columns
    smiles_col = 'processed_smiles'
    target_col = 'Tg'
    
    if smiles_col not in df.columns:
        print(f"   • ERROR: SMILES column '{smiles_col}' not found")
        return None
    if target_col not in df.columns:
        print(f"   • ERROR: Target column '{target_col}' not found")
        return None
    
    # Clean data
    clean_df = df.dropna(subset=[smiles_col, target_col])
    print(f"   • Samples after removing NaN: {len(clean_df)}")
    
    # 2. Target Analysis
    tg_values = clean_df[target_col].values
    print(f"\n2. TARGET ANALYSIS (Tg values):")
    print(f"   • Mean: {np.mean(tg_values):.2f}")
    print(f"   • Std: {np.std(tg_values):.2f}")
    print(f"   • Min: {np.min(tg_values):.2f}")
    print(f"   • Max: {np.max(tg_values):.2f}")
    print(f"   • Range: {np.max(tg_values) - np.min(tg_values):.2f}")
    
    # 3. SMILES Analysis
    smiles_lengths = clean_df[smiles_col].str.len()
    print(f"\n3. SMILES ANALYSIS:")
    print(f"   • Avg SMILES length: {smiles_lengths.mean():.1f}")
    print(f"   • Min SMILES length: {smiles_lengths.min()}")
    print(f"   • Max SMILES length: {smiles_lengths.max()}")
    
    # 4. Model Analysis (if model exists)
    try:
        model = PolymerGCN.load_from_checkpoint(model_path)
        print(f"\n4. MODEL ANALYSIS:")
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"   • Total parameters: {total_params:,}")
        print(f"   • Trainable parameters: {trainable_params:,}")
        print(f"   • Model type: {type(model).__name__}")
        
        # List model components
        print(f"   • Model components:")
        for name, module in model.named_children():
            print(f"     - {name}: {type(module).__name__}")
            
        return {
            'dataset_size': len(clean_df),
            'target_stats': {
                'mean': float(np.mean(tg_values)),
                'std': float(np.std(tg_values)),
                'min': float(np.min(tg_values)),
                'max': float(np.max(tg_values))
            },
            'model': model,
            'data': clean_df
        }
        
    except Exception as e:
        print(f"\n4. MODEL ANALYSIS:")
        print(f"   • ERROR loading model: {str(e)}")
        return {
            'dataset_size': len(clean_df),
            'target_stats': {
                'mean': float(np.mean(tg_values)),
                'std': float(np.std(tg_values)),
                'min': float(np.min(tg_values)),
                'max': float(np.max(tg_values))
            },
            'model': None,
            'data': clean_df
        }
